# Actionability audit notebook (function-by-function)

This notebook is designed to help you **audit and explain** how the actionability algorithm works.

## What you’ll get
- One cell per function used in the actionability pipeline
- Each cell *runs the function* on the same small test set (EN/PT/ES translations)
- Cells print intermediate signals in a readable format to help spot:
  - false positives (scores too high)
  - false negatives (scores too low)
  - score inflation drivers (e.g., keyword hits, tense penalty, SRL-lite flags)

## Data
We use `test_action.csv` from the project root.

In [4]:
# 1) Imports + loading test data (robust parser)
from __future__ import annotations

from pathlib import Path
import pandas as pd

# Notebook lives in nlp/, so project root is one level up
PROJECT_ROOT = Path.cwd().parent
TEST_PATH = PROJECT_ROOT / 'test_action.csv'

print('Project root:', PROJECT_ROOT)
print('Test CSV path:', TEST_PATH)


raw = TEST_PATH.read_text(encoding='utf-8', errors='ignore')

rows = []
for n, line in enumerate(raw.splitlines()):
    line = line.strip()
    if not line:
        continue
    # Skip header line
    if line.lower().startswith('text') and 'lang' in line.lower():
        continue

    # Prefer tab split if present
    if '\t' in line:
        parts = [p.strip() for p in line.split('\t') if p.strip()]
    else:
        # Fallback: split on 2+ spaces (common when exported from editors)
        import re
        parts = [p.strip() for p in re.split(r'\s{2,}', line) if p.strip()]

    if len(parts) < 2:
        continue

    text = parts[0].strip().strip('"')
    lang = parts[-1].strip()

    if lang not in {'en', 'es', 'pt'}:
        continue

    rows.append({'clean_text': text, 'language': lang})

df = pd.DataFrame(rows)

print('Loaded rows:', len(df))
print('Language distribution:', df['language'].value_counts().to_dict())
display(df)

# Helpful sanity: show the 3 intended groups (routes / fatalities / advice)
print('\nSample texts:')
for i, r in df.head(9).iterrows():
    print(f"[{i}] ({r['language']}) {r['clean_text']}")


Project root: /Users/joanacaseiro/Desktop/Year 2/Flood/NLP-model
Test CSV path: /Users/joanacaseiro/Desktop/Year 2/Flood/NLP-model/test_action.csv
Loaded rows: 9
Language distribution: {'en': 3, 'pt': 3, 'es': 3}


,clean_text,language
0,"The tunnel is flooded, take an alternative route.",en
1,"The flood took place at 2am, there are 40 dead...",en
2,If you live in Amsterdam you should avoid leav...,en
3,"O túnel está inundado, utilize uma rota altern...",pt
4,"A inundação ocorreu às 2h, há 40 mortos, eles ...",pt
5,"Se você mora em Amsterdão, deve evitar sair de...",pt
6,"El túnel está inundado, tome una ruta alternat...",es
7,"La inundación ocurrió a las 2am, hay 40 person...",es
8,"Si vive en Ámsterdam, debe evitar salir de su ...",es



Sample texts:
[0] (en) The tunnel is flooded, take an alternative route.
[1] (en) The flood took place at 2am, there are 40 dead people, they were trapped.
[2] (en) If you live in Amsterdam you should avoid leaving your house, instead close all your windows and check for leaks.
[3] (pt) O túnel está inundado, utilize uma rota alternativa.
[4] (pt) A inundação ocorreu às 2h, há 40 mortos, eles ficaram presos.
[5] (pt) Se você mora em Amsterdão, deve evitar sair de casa; em vez disso, feche todas as janelas e verifique se há vazamentos.
[6] (es) El túnel está inundado, tome una ruta alternativa.
[7] (es) La inundación ocurrió a las 2am, hay 40 personas muertas, quedaron atrapadas.
[8] (es) Si vive en Ámsterdam, debe evitar salir de su casa; en su lugar, cierre todas las ventanas y verifique si hay filtraciones.


In [5]:
# 2) Imports from the pipeline
# Ensure the project root is on sys.path so `config.*` and `nlp.*` imports work
import sys
from pathlib import Path
import importlib
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

config = importlib.import_module('config.nlp_config')
actionability = importlib.import_module('nlp.actionability')

print('Project root (added to sys.path):', PROJECT_ROOT)
print('Supported languages:', config.SUPPORTED_LANGUAGES)
print('SpaCy models:', config.SPACY_MODELS)
print('Keyword dict languages:', list(config.ACTIONABILITY_KEYWORDS.keys()))

# The pipeline expects columns: clean_text, language
assert {'clean_text', 'language'}.issubset(df.columns)


Project root (added to sys.path): /Users/joanacaseiro/Desktop/Year 2/Flood/NLP-model
Supported languages: ['en', 'es', 'pt']
SpaCy models: {'en': 'en_core_web_sm', 'es': 'es_core_news_sm', 'pt': 'pt_core_news_sm'}
Keyword dict languages: ['en', 'es', 'pt']


In [6]:
# 3) Function: _truncate_at_sentence(text)
truncate = actionability.split_into_sentences

for i, r in df.iterrows():
    original = r['clean_text']
    truncated = truncate(original)
    print(f'--- row {i} | lang={r["language"]} | original_len={len(original)} | truncated_len={len(truncated)} ---')
    print(truncated)
    print()


--- row 0 | lang=en | original_len=49 | truncated_len=1 ---
['The tunnel is flooded, take an alternative route.']

--- row 1 | lang=en | original_len=73 | truncated_len=1 ---
['The flood took place at 2am, there are 40 dead people, they were trapped.']

--- row 2 | lang=en | original_len=113 | truncated_len=1 ---
['If you live in Amsterdam you should avoid leaving your house, instead close all your windows and check for leaks.']

--- row 3 | lang=pt | original_len=52 | truncated_len=1 ---
['O túnel está inundado, utilize uma rota alternativa.']

--- row 4 | lang=pt | original_len=61 | truncated_len=1 ---
['A inundação ocorreu às 2h, há 40 mortos, eles ficaram presos.']

--- row 5 | lang=pt | original_len=119 | truncated_len=1 ---
['Se você mora em Amsterdão, deve evitar sair de casa; em vez disso, feche todas as janelas e verifique se há vazamentos.']

--- row 6 | lang=es | original_len=50 | truncated_len=1 ---
['El túnel está inundado, tome una ruta alternativa.']

--- row 7 | lang=es

In [9]:
# 4) Function: _get_spacy(lang)
get_spacy = actionability._get_spacy

for lang in sorted(df['language'].unique().tolist()):
    nlp = get_spacy(lang)
    print(f'lang={lang} -> model={config.SPACY_MODELS.get(lang)} -> loaded={nlp is not None}')
    if nlp is not None:
        print('  pipe:', nlp.pipe_names)
    print()
print(df['language'].unique().tolist())


lang=en -> model=en_core_web_sm -> loaded=True
  pipe: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

lang=es -> model=es_core_news_sm -> loaded=True
  pipe: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

lang=pt -> model=pt_core_news_sm -> loaded=True
  pipe: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']

['en', 'pt', 'es']


In [5]:
# 5) Function: score_actionability_keywords(text, lang)
score_kw = actionability.score_actionability_keywords

out = []
for i, r in df.iterrows():
    res = score_kw(r['clean_text'], r['language'])
    out.append({'row': i, 'language': r['language'], **res})

df_kw = pd.DataFrame(out)
display(df_kw)

print('Score formula: total = (imp*0.35) + (short*0.30) + (long*0.15) + (spatial*0.20)')


,row,language,imperative_score,short_term_score,long_term_score,spatial_score,actionability_score
0,0,en,0,0,0,0,0.00
1,1,en,0,0,0,0,0.00
2,2,en,1,0,0,0,0.35
3,3,pt,0,0,0,0,0.00
4,4,pt,0,0,0,0,0.00
5,5,pt,2,0,0,0,0.70
6,6,es,0,0,0,0,0.00
7,7,es,0,0,0,0,0.00
8,8,es,1,0,0,0,0.35


Score formula: total = (imp*0.35) + (short*0.30) + (long*0.15) + (spatial*0.20)


In [6]:
# 6) Function: extract_srl_features(text, lang)
extract_srl = actionability.extract_srl_features

out = []
for i, r in df.iterrows():
    res = extract_srl(r['clean_text'], r['language'])
    out.append({'row': i, 'language': r['language'], **res})

df_srl = pd.DataFrame(out)
display(df_srl)

print('Interpretation: srl_complete=1 iff (has_agent and has_action and has_location)')


,row,language,has_agent,has_action,has_location,srl_complete
0,0,en,1,1,0,0
1,1,en,1,1,0,0
2,2,en,1,1,1,1
3,3,pt,1,1,0,0
4,4,pt,1,1,0,0
5,5,pt,1,1,1,1
6,6,es,1,0,0,0
7,7,es,1,1,0,0
8,8,es,0,1,1,0


Interpretation: srl_complete=1 iff (has_agent and has_action and has_location)


In [7]:
# 7) Function: extract_named_entities(text, lang)
extract_ner = actionability.extract_named_entities

out = []
for i, r in df.iterrows():
    res = extract_ner(r['clean_text'], r['language'])
    out.append({'row': i, 'language': r['language'], **res})

df_ner = pd.DataFrame(out)
display(df_ner)


,row,language,top_locations,top_orgs
0,0,en,[],[]
1,1,en,[],[]
2,2,en,"[""Amsterdam""]",[]
3,3,pt,[],[]
4,4,pt,[],[]
5,5,pt,"[""Amsterdão""]",[]
6,6,es,[],[]
7,7,es,[],[]
8,8,es,"[""Ámsterdam""]",[]


In [8]:
# 8) Function: count_verb_tenses(text, lang)
count_tenses = actionability.count_verb_tenses

out = []
for i, r in df.iterrows():
    res = count_tenses(r['clean_text'], r['language'])
    out.append({'row': i, 'language': r['language'], **res})

df_tenses = pd.DataFrame(out)
display(df_tenses)


,row,language,past_tense,present_tense,future_tense
0,0,en,1,1,0
1,1,en,3,1,0
2,2,en,0,2,0
3,3,pt,0,2,0
4,4,pt,1,1,0
5,5,pt,0,5,0
6,6,es,0,2,0
7,7,es,2,1,0
8,8,es,0,4,0


In [9]:
# 9) Function: assign_temporal_phase(pub_date, flood_date)
# Hardcode both dates for this audit notebook:
# - pub_date: treat as same day as the flood date
# - flood_date: fixed reference date

from datetime import date

assign_phase = actionability.assign_temporal_phase
FLOOD_DATE = date(2024, 1, 1)
PUB_DATE = FLOOD_DATE

out = []
for i, r in df.iterrows():
    out.append({
        'row': i,
        'language': r['language'],
        'pub_date': PUB_DATE,
        'flood_date': FLOOD_DATE,
        'temporal_phase': assign_phase(PUB_DATE, FLOOD_DATE),
    })

df_phase = pd.DataFrame(out)
display(df_phase)


,row,language,pub_date,flood_date,temporal_phase
0,0,en,2024-01-01,2024-01-01,during
1,1,en,2024-01-01,2024-01-01,during
2,2,en,2024-01-01,2024-01-01,during
3,3,pt,2024-01-01,2024-01-01,during
4,4,pt,2024-01-01,2024-01-01,during
5,5,pt,2024-01-01,2024-01-01,during
6,6,es,2024-01-01,2024-01-01,during
7,7,es,2024-01-01,2024-01-01,during
8,8,es,2024-01-01,2024-01-01,during


In [10]:
# 10) Function: run_actionability(df)
# This runs the full actionability feature extraction + scoring and returns an enriched dataframe.
# For this audit notebook we hardcode pub_date and flood_date (both set to the same day).

from datetime import date

run_actionability = actionability.run_actionability

FLOOD_DATE = date(2024, 1, 1)
PUB_DATE = FLOOD_DATE

df_input = df.copy()
df_input['pub_date'] = PUB_DATE
df_input['flood_date'] = FLOOD_DATE

df_enriched = run_actionability(df_input)

cols_to_show = [
    'clean_text','language',
    'pub_date','flood_date','temporal_phase',
    'keyword_score','srl_complete','past_ratio',
    'actionability_score','is_actionable'
]

# Only show columns that exist (guards against upstream schema changes)
cols_to_show = [c for c in cols_to_show if c in df_enriched.columns]

display(df_enriched[cols_to_show])


,clean_text,language,pub_date,flood_date,temporal_phase,srl_complete,actionability_score
0,"The tunnel is flooded, take an alternative route",en,2024-01-01,2024-01-01,during,0,0.00
1,"The flood took place at 2am, there are 40 dead...",en,2024-01-01,2024-01-01,during,0,0.00
2,If you live in Amsterdam you should avoid leav...,en,2024-01-01,2024-01-01,during,1,0.35
3,"O túnel está inundado, utilize uma rota altern...",pt,2024-01-01,2024-01-01,during,0,0.00
4,"A inundação ocorreu às 2h, há 40 mortos, eles ...",pt,2024-01-01,2024-01-01,during,0,0.00
5,"Se você mora em Amsterdão, deve evitar sair de...",pt,2024-01-01,2024-01-01,during,1,0.70
6,"El túnel está inundado, tome una ruta alternativa",es,2024-01-01,2024-01-01,during,0,0.00
7,"La inundación ocurrió a las 2am, hay 40 person...",es,2024-01-01,2024-01-01,during,0,0.00
8,"Si vive en Ámsterdam, debe evitar salir de su ...",es,2024-01-01,2024-01-01,during,0,0.35
